# Eutectic Si 균형 미세 조정

기존 Eu best.pt와 네 specialist 완료 export에서 시작합니다. Eu만 학습하고 결합 mIoU로 best를 선택합니다.


## 설정


In [ ]:
from pathlib import Path
from google.colab import drive
import os
drive.mount("/content/drive")
PROJECT_ROOT=Path("/content/drive/MyDrive/KoMaP2026")
DATA_ROOT=Path("/content/komap_advanced_ram_data")
NUM_WORKERS=2
RESUME=True
EVAL_EVERY=3
EARLY_STOPPING_PATIENCE=6
MIN_EPOCHS=20
STOP_AFTER_EPOCH=None
LOG_EVERY=100
TTA_ROTATIONS=4
COMPARE_PLAIN_TTA=True
ABSENT_POLICY="ignore"

BASE_RUN_NAMES=[None,None,None,None]
BASE_WEIGHTS=[1.0,1.0,1.0,1.0]
NEW_RUN_NAMES={2:"advanced_eu_balanced_ram_01"}
BATCH_SIZE=8
PATCH_SIZE=640
EPOCHS=30
EARLY_STOPPING_PATIENCE=4
MIN_EPOCHS=12


## 환경·코드


In [ ]:
import os, sys, importlib
from pathlib import Path
_candidates = [Path(os.environ['KOMAP_SUBMISSION_ROOT'])] if os.environ.get('KOMAP_SUBMISSION_ROOT') else []
_candidates += [Path.cwd(), Path('/content/KoMaP2026_Submission_DirectInference_20261015'),
                Path('/content/drive/MyDrive/KoMaP2026_Submission_DirectInference_20261015')]
CODE_ROOT = next((p.resolve() for p in _candidates if (p/'komap_submission_environment.py').is_file() and (p/'komap_submission_infer.py').is_file() and 'CURRENT_EXPORT_SHA256' in (p/'komap_submission_infer.py').read_text(encoding='utf-8')), None)
if CODE_ROOT is None:
    raise FileNotFoundError('제출 코드 폴더를 찾지 못했습니다. 00_제출파일_실행안내.ipynb에서 KOMAP_SUBMISSION_ROOT 경로를 설정하세요.')
import subprocess
subprocess.check_call([sys.executable, str(CODE_ROOT/'komap_submission_environment.py'), '--install'])


## 환경·코드


In [ ]:
import os, sys, importlib
from pathlib import Path
_candidates = [Path(os.environ['KOMAP_SUBMISSION_ROOT'])] if os.environ.get('KOMAP_SUBMISSION_ROOT') else []
_candidates += [Path.cwd(), Path('/content/KoMaP2026_Submission_DirectInference_20261015'),
                Path('/content/drive/MyDrive/KoMaP2026_Submission_DirectInference_20261015')]
CODE_ROOT = next((p.resolve() for p in _candidates if (p/'komap_submission_environment.py').is_file() and (p/'komap_submission_infer.py').is_file() and 'CURRENT_EXPORT_SHA256' in (p/'komap_submission_infer.py').read_text(encoding='utf-8')), None)
if CODE_ROOT is None:
    raise FileNotFoundError('제출 코드 폴더를 찾지 못했습니다. 00_제출파일_실행안내.ipynb에서 KOMAP_SUBMISSION_ROOT 경로를 설정하세요.')
CODE_DIR = CODE_ROOT / '.'
sys.dont_write_bytecode = True

for _module_name in list(sys.modules):
    if _module_name.startswith('komap_'):
        del sys.modules[_module_name]
sys.path[:] = [p for p in sys.path if p not in (str(CODE_ROOT), str(CODE_ROOT/'legacy_yolo_initial'))]
sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
print('제출 코드:', CODE_DIR)


## 데이터·초기 모델


In [ ]:
import komap_specialist_data as kd
import komap_phase_fusion as pf
import komap_efficient_ram as kr
for module in (kd,pf,kr):
    importlib.reload(module)
    if Path(module.__file__).resolve().parent!=CODE_DIR.resolve():
        raise RuntimeError("다른 코드가 로드되었습니다. 새 런타임에서 실행하세요.")

RAM, DATA_REPORT=kr.prepare_ram(PROJECT_ROOT,DATA_ROOT)


## 데이터·초기 모델


In [ ]:
BASE_EXPORTS=pf.resolve_baseline(PROJECT_ROOT,BASE_RUN_NAMES)
FUSION_CONTEXT=pf.fusion_context(BASE_EXPORTS,RAM,BASE_WEIGHTS)
INITIAL_CHECKPOINTS={}
for target in (2,):
    checkpoint=BASE_EXPORTS[target]["run_dir"] / "best.pt"
    expected=BASE_EXPORTS[target]["manifest"]["checkpoint_sha256"]
    if not checkpoint.exists() or kr.file_sha(checkpoint)!=expected:
        raise ValueError(f"클래스 {target}: 기존 export와 일치하는 best.pt가 필요합니다: {checkpoint}")
    INITIAL_CHECKPOINTS[target]=checkpoint
print("기존 best 미세 조정:",INITIAL_CHECKPOINTS)


## 학습·검증


In [ ]:
RESULTS={}
for target in (2,):
    print("\n시작:",kd.CLASS_NAMES[target])
    CONFIG=kr.Config(architecture="specialist",target_class=target,encoder_weights=None,
        patch=PATCH_SIZE,batch=BATCH_SIZE,epochs=EPOCHS,eval_stride=PATCH_SIZE//2,
        encoder_lr=1e-4,decoder_lr=1e-4,freeze_encoder_epochs=0,dice_weight=1.0,
        eval_every=EVAL_EVERY,absent_policy=ABSENT_POLICY)
    RUN_DIR=PROJECT_ROOT/"runs"/NEW_RUN_NAMES[target]
    BEST_PATH=kr.train(RAM,DATA_REPORT,RUN_DIR,CONFIG,num_workers=NUM_WORKERS,
        stop_after_epoch=STOP_AFTER_EPOCH,patience=EARLY_STOPPING_PATIENCE,min_epochs=MIN_EPOCHS,
        log_every=LOG_EVERY,resume=RESUME,initial_checkpoint=INITIAL_CHECKPOINTS[target],fusion_context=FUSION_CONTEXT)
    RESULTS[target]=kr.export_best(RAM,DATA_REPORT,RUN_DIR,tta_rotations=TTA_ROTATIONS,
        compare_plain=COMPARE_PLAIN_TTA,fusion_context=FUSION_CONTEXT)
print("Eu 완료:",RESULTS)
